# Paper 10: The Genome as Knowledge Graph — Governance, Scaling, and the Architecture of Multicellular Life

**Reproducibility notebook** for McCarthy (2026), Paper 10.

Tests the prediction that governance layers (repair/checkpoint systems) scale as
log₁₀(cell count) across organisms. All data is hardcoded from primary literature.
No API calls or external dependencies required.

**Datasets:**
- 10 organisms spanning 15 orders of magnitude of cell count
- 16 mammalian species with measured somatic mutation rates (Cagan et al. 2022, Nature 604:517–524)

In [ ]:
# Cell 1: Setup
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
%matplotlib inline

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10

In [ ]:
# Cell 2: Organism data — 10 species spanning 15 orders of magnitude
#
# Governance layers counted as discrete repair/checkpoint systems:
#   1. Polymerase proofreading (3'→5' exonuclease)
#   2. Mismatch repair (MMR)
#   3. Base excision repair (BER)
#   4. Nucleotide excision repair (NER)
#   5. Double-strand break repair (HR + NHEJ)
#   6. Cell cycle checkpoints + apoptosis (p53/Rb)
#   7. Immune surveillance (NK cells, T-cell tumor recognition)

organisms = {
    "E. coli": {
        "cell_count": 1, "layers": 2,
        "layer_names": ["proofreading", "MMR"],
        "mutation_rate": 5.4e-10, "base_rate_unsuppressed": 1e-6,
        "gen_time_days": 0.014, "offspring_per_gen": 2,
        "lifespan_years": 0.001, "cancer_rate": 0,
        "somatic_mut_per_year": None,
    },
    "S. cerevisiae": {
        "cell_count": 1, "layers": 3,
        "layer_names": ["proofreading", "MMR", "BER"],
        "mutation_rate": 3.3e-10, "base_rate_unsuppressed": 1e-6,
        "gen_time_days": 0.083, "offspring_per_gen": 2,
        "lifespan_years": 0.001, "cancer_rate": 0,
        "somatic_mut_per_year": None,
    },
    "C. elegans": {
        "cell_count": 959, "layers": 5,
        "layer_names": ["proofreading", "MMR", "BER", "NER", "apoptosis"],
        "mutation_rate": 2.7e-9, "base_rate_unsuppressed": 1e-6,
        "gen_time_days": 3, "offspring_per_gen": 300,
        "lifespan_years": 0.05, "cancer_rate": 0,
        "somatic_mut_per_year": None,
    },
    "D. melanogaster": {
        "cell_count": 5e4, "layers": 5,
        "layer_names": ["proofreading", "MMR", "BER", "NER", "apoptosis"],
        "mutation_rate": 3.5e-9, "base_rate_unsuppressed": 1e-6,
        "gen_time_days": 10, "offspring_per_gen": 400,
        "lifespan_years": 0.16, "cancer_rate": 0.01,
        "somatic_mut_per_year": None,
    },
    "Zebrafish": {
        "cell_count": 1e10, "layers": 6,
        "layer_names": ["proofreading", "MMR", "BER", "NER", "DSB", "checkpoints"],
        "mutation_rate": 7.5e-9, "base_rate_unsuppressed": 1e-6,
        "gen_time_days": 90, "offspring_per_gen": 200,
        "lifespan_years": 3.5, "cancer_rate": 0.05,
        "somatic_mut_per_year": None,
    },
    "Naked mole rat": {
        "cell_count": 1e10, "layers": 7,
        "layer_names": ["proofreading", "MMR", "BER", "NER", "DSB", "checkpoints+", "immune+"],
        "mutation_rate": 1e-9, "base_rate_unsuppressed": 1e-6,
        "gen_time_days": 365, "offspring_per_gen": 12,
        "lifespan_years": 37, "cancer_rate": 0.001,
        "somatic_mut_per_year": 93,
    },
    "Mouse": {
        "cell_count": 1e10, "layers": 7,
        "layer_names": ["proofreading", "MMR", "BER", "NER", "DSB", "checkpoints", "immune"],
        "mutation_rate": 5.4e-9, "base_rate_unsuppressed": 1e-6,
        "gen_time_days": 60, "offspring_per_gen": 8,
        "lifespan_years": 2, "cancer_rate": 0.30,
        "somatic_mut_per_year": 796,
    },
    "Human": {
        "cell_count": 3.7e13, "layers": 7,
        "layer_names": ["proofreading", "MMR", "BER", "NER", "DSB", "checkpoints", "immune"],
        "mutation_rate": 5e-10, "base_rate_unsuppressed": 1e-6,
        "gen_time_days": 5475, "offspring_per_gen": 1,
        "lifespan_years": 80, "cancer_rate": 0.40,
        "somatic_mut_per_year": 47,
    },
    "Elephant": {
        "cell_count": 1e15, "layers": 7,
        "layer_names": ["proofreading", "MMR", "BER", "NER", "DSB", "checkpoints+", "immune"],
        "mutation_rate": 3e-10, "base_rate_unsuppressed": 1e-6,
        "gen_time_days": 8030, "offspring_per_gen": 1,
        "lifespan_years": 65, "cancer_rate": 0.05,
        "somatic_mut_per_year": None,
    },
    "Bowhead whale": {
        "cell_count": 1e15, "layers": 7,
        "layer_names": ["proofreading", "MMR", "BER", "NER", "DSB+", "checkpoints", "immune"],
        "mutation_rate": 2e-10, "base_rate_unsuppressed": 1e-6,
        "gen_time_days": 7300, "offspring_per_gen": 1,
        "lifespan_years": 200, "cancer_rate": 0.02,
        "somatic_mut_per_year": None,
    },
}

print(f"{len(organisms)} organisms loaded.")
for name, o in organisms.items():
    print(f"  {name:20s}  cells={o['cell_count']:.0e}  layers={o['layers']}  rate={o['mutation_rate']:.1e}")

In [ ]:
# Cell 3: Cagan et al. 2022 — somatic mutation rates across 16 mammals
# Source: Cagan et al., Nature 604:517-524 (2022), Table 1

cagan_species = {
    "Mouse":               {"mut_per_year": 796,  "lifespan": 2.7},
    "Rat":                 {"mut_per_year": 397,  "lifespan": 3.5},
    "Rabbit":              {"mut_per_year": 172,  "lifespan": 9},
    "Naked mole rat":      {"mut_per_year": 93,   "lifespan": 37},
    "Dog":                 {"mut_per_year": 249,  "lifespan": 12},
    "Ferret":              {"mut_per_year": 196,  "lifespan": 8},
    "Cat":                 {"mut_per_year": 120,  "lifespan": 15},
    "Lion":                {"mut_per_year": 160,  "lifespan": 15},
    "Tiger":               {"mut_per_year": 76,   "lifespan": 20},
    "Ring-tailed lemur":   {"mut_per_year": 117,  "lifespan": 20},
    "Human":               {"mut_per_year": 47,   "lifespan": 80},
    "Harbour porpoise":    {"mut_per_year": 63,   "lifespan": 24},
    "Cow":                 {"mut_per_year": 67,   "lifespan": 20},
    "Horse":               {"mut_per_year": 60,   "lifespan": 30},
    "Giraffe":             {"mut_per_year": 99,   "lifespan": 25},
    "Black-and-white colobus": {"mut_per_year": 151, "lifespan": 20},
}

print(f"{len(cagan_species)} species loaded from Cagan et al. 2022.")

In [ ]:
# Cell 4: Primary fit — layers vs log₁₀(cell count)

names = list(organisms.keys())
cell_counts = np.array([organisms[o]["cell_count"] for o in names])
layers = np.array([organisms[o]["layers"] for o in names])
mutation_rates = np.array([organisms[o]["mutation_rate"] for o in names])
base_rate = 1e-6

log_cell_count = np.log10(np.maximum(cell_counts, 1))
suppression_orders = np.log10(base_rate / mutation_rates)

slope, intercept, r_value, p_value, std_err = stats.linregress(log_cell_count, layers)

print("=" * 70)
print("GOVERNANCE SCALING: layers vs log₁₀(cell count)")
print("=" * 70)
print(f"\nLinear fit: layers = {slope:.3f} × log₁₀(n) + {intercept:.3f}")
print(f"R² = {r_value**2:.4f}")
print(f"p  = {p_value:.2e}")
print(f"slope SE = {std_err:.3f}")
print()

for name in names:
    o = organisms[name]
    lcc = np.log10(max(o["cell_count"], 1))
    predicted = slope * lcc + intercept
    print(f"  {name:20s}  cells={o['cell_count']:.0e}  log₁₀={lcc:5.1f}  "
          f"layers={o['layers']}  predicted={predicted:.1f}  rate={o['mutation_rate']:.1e}")

In [ ]:
# Cell 5: Suppression scaling — log₁₀(base/observed) vs log₁₀(cell count)

slope2, intercept2, r2, p2, se2 = stats.linregress(log_cell_count, suppression_orders)

print("=" * 70)
print("SUPPRESSION SCALING: log₁₀(base/observed) vs log₁₀(cell count)")
print("=" * 70)
print(f"\nLinear fit: suppression = {slope2:.3f} × log₁₀(n) + {intercept2:.3f}")
print(f"R² = {r2**2:.4f}")
print(f"p  = {p2:.2e}")
print()

for name in names:
    o = organisms[name]
    lcc = np.log10(max(o["cell_count"], 1))
    supp = np.log10(base_rate / o["mutation_rate"])
    print(f"  {name:20s}  log₁₀(n)={lcc:5.1f}  suppression={supp:.1f} orders")

In [ ]:
# Cell 6: Peto's paradox check — mutation_rate × cell_count

print("=" * 70)
print("PETO'S PARADOX CHECK: mutation_rate × cell_count")
print("=" * 70)
print()

for name in names:
    o = organisms[name]
    burden = o["mutation_rate"] * o["cell_count"]
    print(f"  {name:20s}  rate×n = {burden:.2e}")

print()
print("If governance scales correctly, burden should NOT scale linearly with n.")
print("Elephant burden should be comparable to mouse, not 10⁵ × mouse.")

In [ ]:
# Cell 7: Figure 1 — Governance scaling (3 panels)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: Layers vs log(cell count)
ax = axes[0]
ax.scatter(log_cell_count, layers, s=80, c='steelblue', zorder=5)
for i, name in enumerate(names):
    ax.annotate(name, (log_cell_count[i], layers[i]),
                textcoords="offset points", xytext=(5, 5), fontsize=8)
x_fit = np.linspace(-0.5, 16, 100)
ax.plot(x_fit, slope * x_fit + intercept, 'r--', alpha=0.7,
        label=f'y = {slope:.2f}x + {intercept:.2f}\nR² = {r_value**2:.3f}, p = {p_value:.1e}')
ax.set_xlabel('log₁₀(cell count)')
ax.set_ylabel('Governance layers')
ax.set_title('A. Repair layers scale as log(n)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel B: Suppression orders vs log(cell count)
ax = axes[1]
ax.scatter(log_cell_count, suppression_orders, s=80, c='darkgreen', zorder=5)
for i, name in enumerate(names):
    ax.annotate(name, (log_cell_count[i], suppression_orders[i]),
                textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.plot(x_fit, slope2 * x_fit + intercept2, 'r--', alpha=0.7,
        label=f'y = {slope2:.2f}x + {intercept2:.2f}\nR² = {r2**2:.3f}')
ax.set_xlabel('log₁₀(cell count)')
ax.set_ylabel('Orders of magnitude suppressed')
ax.set_title('B. Mutation rate suppression vs scale')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel C: Burden with/without governance
ax = axes[2]
n_range = np.logspace(0, 16, 200)
burden_log = base_rate * n_range / (10 ** (0.35 * np.log10(np.maximum(n_range, 1))))
burden_none = base_rate * n_range

ax.loglog(n_range, burden_none, 'r-', alpha=0.5, label='No governance (bacterial rate × n)')
ax.loglog(n_range, burden_log, 'b-', alpha=0.7, label='Log governance (observed)')

actual_burden = mutation_rates * cell_counts
ax.scatter(cell_counts, actual_burden, s=80, c='black', zorder=5)
for i, name in enumerate(names):
    ax.annotate(name, (cell_counts[i], actual_burden[i]),
                textcoords="offset points", xytext=(5, 5), fontsize=7)

ax.set_xlabel('Cell count')
ax.set_ylabel('Mutation burden (rate × n)')
ax.set_title('C. Governance prevents linear burden scaling')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Confound analysis — 4 candidate predictors

repro_rates = np.array([organisms[o]["offspring_per_gen"] / organisms[o]["gen_time_days"] for o in names])
gen_times = np.array([organisms[o]["gen_time_days"] for o in names])
lifespans = np.array([organisms[o]["lifespan_years"] for o in names])

log_repro = np.log10(repro_rates + 1e-10)
log_gen = np.log10(gen_times + 1e-10)
log_lifespan = np.log10(lifespans + 1e-10)

fit_cell = stats.linregress(log_cell_count, layers)
fit_repro = stats.linregress(log_repro, layers)
fit_gen = stats.linregress(log_gen, layers)
fit_lifespan = stats.linregress(log_lifespan, layers)

print("=" * 70)
print("CONFOUND ANALYSIS: What drives governance layers?")
print("=" * 70)
print()
print(f"{'Predictor':30s}  {'R²':>8s}  {'p-value':>10s}  {'slope':>8s}")
print("-" * 65)
print(f"{'log₁₀(cell count)':30s}  {fit_cell.rvalue**2:8.4f}  {fit_cell.pvalue:10.2e}  {fit_cell.slope:8.3f}")
print(f"{'log₁₀(repro rate)':30s}  {fit_repro.rvalue**2:8.4f}  {fit_repro.pvalue:10.2e}  {fit_repro.slope:8.3f}")
print(f"{'log₁₀(generation time)':30s}  {fit_gen.rvalue**2:8.4f}  {fit_gen.pvalue:10.2e}  {fit_gen.slope:8.3f}")
print(f"{'log₁₀(lifespan)':30s}  {fit_lifespan.rvalue**2:8.4f}  {fit_lifespan.pvalue:10.2e}  {fit_lifespan.slope:8.3f}")

In [ ]:
# Cell 9: Key counterexample — Mouse vs Naked mole rat

print("=" * 70)
print("KEY COUNTEREXAMPLE: Naked mole rat vs Mouse")
print("=" * 70)
print()
print(f"{'Metric':30s}  {'Mouse':>12s}  {'Naked mole rat':>15s}")
print("-" * 62)

for metric, key, fmt in [
    ("Cell count", "cell_count", ".0e"),
    ("Governance layers", "layers", "d"),
    ("Mutation rate (per bp/div)", "mutation_rate", ".1e"),
    ("Generation time (days)", "gen_time_days", ".0f"),
    ("Offspring per generation", "offspring_per_gen", "d"),
    ("Lifespan (years)", "lifespan_years", ".0f"),
    ("Cancer rate", "cancer_rate", ".3f"),
    ("Somatic mutations/year", "somatic_mut_per_year", ".0f"),
]:
    m = organisms["Mouse"][key]
    n = organisms["Naked mole rat"][key]
    if m is not None and n is not None:
        print(f"  {metric:28s}  {m:>12{fmt}}  {n:>15{fmt}}")

print()
print("Same cell count. Same layer count.")
print("Mouse: 30% cancer, 2yr lifespan, 796 mut/yr.")
print("Naked mole rat: <0.1% cancer, 37yr lifespan, 93 mut/yr.")
print()
print("→ Layers track cell count (architecture).")
print("→ Cancer rate tracks layer QUALITY (confidence per layer).")

In [ ]:
# Cell 10: Figure 2 — Confound analysis (4 panels)

fig2, axes2 = plt.subplots(2, 2, figsize=(12, 10))

predictors = [
    (log_cell_count, "log₁₀(cell count)", fit_cell, "steelblue"),
    (log_repro, "log₁₀(reproduction rate)", fit_repro, "darkred"),
    (log_gen, "log₁₀(generation time, days)", fit_gen, "darkgreen"),
    (log_lifespan, "log₁₀(lifespan, years)", fit_lifespan, "purple"),
]

for ax, (x, xlabel, fit, color) in zip(axes2.flat, predictors):
    ax.scatter(x, layers, s=80, c=color, zorder=5)
    for i, name in enumerate(names):
        ax.annotate(name, (x[i], layers[i]),
                    textcoords="offset points", xytext=(5, 5), fontsize=7)
    xfit = np.linspace(x.min() - 0.5, x.max() + 0.5, 100)
    ax.plot(xfit, fit.slope * xfit + fit.intercept, 'r--', alpha=0.7)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Governance layers")
    ax.set_title(f"R² = {fit.rvalue**2:.3f}, p = {fit.pvalue:.1e}")
    ax.grid(True, alpha=0.3)

    # Highlight naked mole rat and mouse
    for special in ["Naked mole rat", "Mouse"]:
        idx = names.index(special)
        ax.scatter(x[idx], layers[idx], s=120, facecolors='none',
                   edgecolors='red', linewidths=2, zorder=6)

fig2.suptitle("Confound analysis: What predicts governance layers?\n"
              "(Red circles = Mouse & Naked mole rat — same cells, same layers, different reproduction)",
              fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11: Cagan analysis — mutation rate × lifespan ≈ constant

cagan_names = list(cagan_species.keys())
cagan_mut = np.array([cagan_species[s]["mut_per_year"] for s in cagan_names])
cagan_life = np.array([cagan_species[s]["lifespan"] for s in cagan_names])
cagan_burden = cagan_mut * cagan_life

print("=" * 70)
print("CAGAN ANALYSIS: Somatic mutation rate × lifespan")
print("Source: Cagan et al. 2022, Nature 604:517-524")
print("=" * 70)
print()
print(f"{'Species':30s}  {'Mut/yr':>8s}  {'Lifespan':>8s}  {'Burden':>10s}")
print("-" * 62)
for i, name in enumerate(cagan_names):
    print(f"  {name:28s}  {cagan_mut[i]:>8.0f}  {cagan_life[i]:>8.1f}  {cagan_burden[i]:>10.0f}")

print()
print(f"  Mean lifetime burden:   {np.mean(cagan_burden):>10.0f} mutations")
print(f"  Median lifetime burden: {np.median(cagan_burden):>10.0f} mutations")
print(f"  Std:                    {np.std(cagan_burden):>10.0f}")
print(f"  CV (std/mean):          {np.std(cagan_burden)/np.mean(cagan_burden):>10.3f}")

log_mut = np.log10(cagan_mut)
log_life = np.log10(cagan_life)

fit_cagan = stats.linregress(log_life, log_mut)
print(f"\nlog₁₀(mut/yr) = {fit_cagan.slope:.3f} × log₁₀(lifespan) + {fit_cagan.intercept:.3f}")
print(f"R² = {fit_cagan.rvalue**2:.4f}")
print(f"p  = {fit_cagan.pvalue:.2e}")
print(f"Slope = {fit_cagan.slope:.3f} (prediction: -1.0 if burden is constant)")

fit_burden_life = stats.linregress(cagan_life, cagan_burden)
print(f"\nBurden vs lifespan: slope = {fit_burden_life.slope:.1f}, "
      f"R² = {fit_burden_life.rvalue**2:.4f}, p = {fit_burden_life.pvalue:.2e}")
if fit_burden_life.pvalue > 0.05:
    print("→ Burden is NOT significantly correlated with lifespan (p > 0.05)")
    print("→ CONFIRMED: lifetime mutation burden is approximately constant across mammals")

In [ ]:
# Cell 12: Figure 3 — The kill shot (3 panels)

fig3, axes3 = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: log(mut/yr) vs log(lifespan)
ax = axes3[0]
ax.scatter(log_life, log_mut, s=80, c='crimson', zorder=5)
for i, name in enumerate(cagan_names):
    ax.annotate(name, (log_life[i], log_mut[i]),
                textcoords="offset points", xytext=(5, 5), fontsize=7)
xfit = np.linspace(log_life.min() - 0.2, log_life.max() + 0.2, 100)
ax.plot(xfit, fit_cagan.slope * xfit + fit_cagan.intercept, 'k--', alpha=0.7,
        label=f'slope = {fit_cagan.slope:.2f}\nR² = {fit_cagan.rvalue**2:.3f}')
ax.plot(xfit, -1.0 * xfit + fit_cagan.intercept, 'b:', alpha=0.4,
        label='slope = -1.0 (perfect constant burden)')
ax.set_xlabel('log₁₀(lifespan, years)')
ax.set_ylabel('log₁₀(somatic mutations per year)')
ax.set_title('A. Mutation rate inversely predicts lifespan')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel B: Lifetime burden
ax = axes3[1]
ax.scatter(cagan_life, cagan_burden, s=80, c='darkgreen', zorder=5)
for i, name in enumerate(cagan_names):
    ax.annotate(name, (cagan_life[i], cagan_burden[i]),
                textcoords="offset points", xytext=(5, 5), fontsize=7)
ax.axhline(np.mean(cagan_burden), color='red', linestyle='--', alpha=0.7,
           label=f'Mean = {np.mean(cagan_burden):.0f} (CV = {np.std(cagan_burden)/np.mean(cagan_burden):.2f})')
ax.fill_between([0, cagan_life.max() * 1.1],
                np.mean(cagan_burden) - np.std(cagan_burden),
                np.mean(cagan_burden) + np.std(cagan_burden),
                alpha=0.15, color='red')
ax.set_xlabel('Lifespan (years)')
ax.set_ylabel('Lifetime mutation burden\n(mutations/yr × lifespan)')
ax.set_title('B. Lifetime burden is approximately constant')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, cagan_life.max() * 1.1)

# Panel C: Confidence per layer
ax = axes3[2]
synth_names_list, synth_conf, synth_cancer, synth_lifespan_list = [], [], [], []
for name in names:
    o = organisms[name]
    if o.get("somatic_mut_per_year") is not None and o["cancer_rate"] > 0:
        synth_names_list.append(name)
        conf = np.log10(1e-6 / o["mutation_rate"]) / o["layers"]
        synth_conf.append(conf)
        synth_cancer.append(o["cancer_rate"])
        synth_lifespan_list.append(o["lifespan_years"])

synth_conf = np.array(synth_conf)
synth_cancer = np.array(synth_cancer)
synth_lifespan_arr = np.array(synth_lifespan_list)

ax.scatter(synth_conf, synth_lifespan_arr, s=100, c='steelblue', zorder=5)
for i, name in enumerate(synth_names_list):
    ax.annotate(name, (synth_conf[i], synth_lifespan_arr[i]),
                textcoords="offset points", xytext=(5, 5), fontsize=9, fontweight='bold')
    cancer_color = plt.cm.Reds(synth_cancer[i] / max(synth_cancer))
    ax.scatter(synth_conf[i], synth_lifespan_arr[i], s=200,
               facecolors='none', edgecolors=cancer_color, linewidths=2, zorder=4)

ax.set_xlabel('Confidence per layer\n(orders of suppression / layers)')
ax.set_ylabel('Lifespan (years)')
ax.set_title('C. Confidence per layer predicts lifespan\n(red ring intensity = cancer rate)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()